In [0]:
print("dq_check_started")

from pyspark.sql import functions as f
from datetime import datetime

In [0]:
%sql
USE CATALOG marketing_analytics_capstone;
create schema if not exists monitoring;

In [0]:
S3_BUCKET               = "s3://marketing-analytics-capstone"
S3_QUARANTINE_PATH      = f"{S3_BUCKET}/silver/quarantine"
S3_METRICS_PATH         = f"{S3_BUCKET}/monitoring/silver_data_quality_metrics"
S3_CHECKPOINT_PATH      = f"{S3_BUCKET}/checkpoints/silver"
MAX_BAD_PERCENTAGE = 0.05   # 5%
MAX_BAD_RECORDS = 50   

In [0]:
df_stream = (
    spark.readStream
    .table("marketing_analytics_capstone.silver.silver_campaigns")
)

In [0]:
from pyspark.sql import functions as f
from datetime import datetime

def dq_check(batch_df, batch_id):

    metrics = batch_df.agg(
        f.count("*").alias("total_records"),
        f.sum(f.when(~f.col("dq_pass"), 1).otherwise(0)).alias("bad_records")
    ).collect()[0]

    total_records = metrics["total_records"]
    bad_records = metrics["bad_records"]
    bad_percentage = bad_records / total_records if total_records > 0 else 0

    # ---- Bad records ----
    df_bad = batch_df.filter("dq_pass = false").withColumn(
        "dq_failure_reason",
        f.concat_ws(",",
            f.when(~f.col("dq_valid_cost"), f.lit("invalid_cost")),
            f.when(~f.col("dq_valid_date"), f.lit("invalid_date")),
            f.when(~f.col("dq_valid_impressions"), f.lit("invalid_impressions")),
            f.when(~f.col("dq_valid_clicks"), f.lit("invalid_clicks")),
            f.when(~f.col("dq_valid_conversion_rate"), f.lit("invalid_conversion_rate"))
        )
    )

    # ---- Write to S3 (IMPORTANT CHANGE) ----
    df_bad.write \
    .format("delta") \
    .mode("append") \
    .option("path", f"{S3_QUARANTINE_PATH}/") \
    .saveAsTable("marketing_analytics_capstone.silver.quarantine_campaigns")

    # ---- Metrics ----
    from pyspark.sql import Row
    metrics_row = [Row(
        run_timestamp=datetime.now(),
        total_records=total_records,
        bad_records=bad_records,
        bad_percentage=bad_percentage
    )]

    spark.createDataFrame(metrics_row).write \
    .format("delta") \
    .mode("append").save(f"{S3_METRICS_PATH}/")
    # .option("path", f"{S3_METRICS_PATH}/") \
    # .saveAsTable("marketing_analytics_capstone.monitoring.silver_metrics")
    

    # ---- Threshold check ----
    if bad_percentage > MAX_BAD_PERCENTAGE or bad_records > MAX_BAD_RECORDS:
        raise Exception(f"DQ FAILED for batch {batch_id}")
    else:
        print(f"DQ passed for batch {batch_id}")


In [0]:
query = (
    df_stream.writeStream
    .foreachBatch(dq_check)
    .option("checkpointLocation", f"{S3_CHECKPOINT_PATH}/silver_dq_camp/checkpoint/")
    .trigger(availableNow=True) 
    .start()
)

query.awaitTermination()
print("dq_check_completed")

In [0]:
%sql
select count(*) from marketing_analytics_capstone.silver.quarantine_campaigns;